# 18 — Unified Recommender Signal Analysis (diagnostics)

Signal-level diagnostics of the nine-signal unified recommender. Read **after** the headline baseline comparison in NB 17. These plots explain *why* NB 17's numbers are what they are; they do not establish the numbers themselves.

Five diagnostics:
1. Data characteristics and evaluation ceiling
2. Signal weight sensitivity (11 configurations)
3. Split ratio sensitivity with eligibility thresholds
4. Signal contribution profiling
5. Per-signal ablation (drop-one analysis)

Output weight configurations here feed NB 17 (`ROUTE_HEAVY_WEIGHTS` etc).

In [ ]:
import sys
import time

sys.path.insert(0, '..')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from helpers.unified_recommender import UnifiedRecommender, UNIFIED_WEIGHTS

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

OUTPUT_DIR = Path('./output/evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Imports ready.')
print(f'Default unified weights: {UNIFIED_WEIGHTS}')

In [ ]:
# Load unified recommender
FEATURES_PATH = './output/semantic_features/trajectory_features.json'
t0 = time.time()
urec = UnifiedRecommender()
urec.load_all_data(features_path=FEATURES_PATH)
print(f'Load time: {time.time()-t0:.1f}s')

## 1. Data Characteristics & Evaluation Ceiling

Before interpreting any metric, understand the structural constraints of the data.

In [ ]:
# Sequence length distribution
lengths = [len(s) for s in urec.sequences.values()]
thresholds = [2, 3, 4, 5]

print('=== Trajectory Sequence Lengths ===')
print(f'  Total trajectories: {len(lengths)}')
print(f'  Mean: {np.mean(lengths):.1f}, Median: {np.median(lengths):.0f}, Max: {max(lengths)}')
for t in thresholds:
    n = sum(1 for l in lengths if l >= t)
    print(f'  With >={t} POIs: {n} ({n/len(lengths)*100:.1f}%)')

print('\n  Length distribution:')
for length, count in sorted(zip(*np.unique(lengths, return_counts=True))):
    bar = '#' * count
    print(f'    {length:2d}: {count:3d} {bar}')

# Holdout recoverability analysis
print('\n=== Holdout Recoverability ===')
for min_len in [3, 5]:
    eligible = [t for t in urec.sequences if len(urec.sequences[t]) >= min_len]
    total_holdout = 0
    recoverable = 0
    for t in eligible:
        seq = urec.sequences[t]
        split_idx = max(2, int(len(seq) * 0.7))
        holdout = set(seq[split_idx:])
        total_holdout += len(holdout)
        other_pois = set()
        for other_t, pois in urec.traj_poi_sets.items():
            if other_t != t:
                other_pois |= pois
        recoverable += len(holdout & other_pois)
    pct = recoverable / total_holdout * 100 if total_holdout else 0
    print(f'  min_len={min_len}: {len(eligible)} trajs, '
          f'{total_holdout} holdout POIs, '
          f'{recoverable} recoverable ({pct:.1f}%), '
          f'{total_holdout - recoverable} unique ({100-pct:.1f}%)')

# Total POI pool
print(f'\n  Total POI pool: {len(urec.all_poi_uris)}')
print(f'  Random baseline R@10: {10/len(urec.all_poi_uris)*100:.2f}%')

## 2. Signal Weight Sensitivity

11 weight configurations testing different hypotheses about which evidence types drive recommendation quality.

Evaluated with `min_seq_length=3` (larger N) and `leave_k_out` (k=1) + `trajectory_split` (70/30) protocols.

In [ ]:
weight_configs = {
    'default': {
        'archetype_sim': 0.15, 'semantic_sim': 0.17, 'route_sim': 0.22, 'flow_prob': 0.05,
        'dominant_ctx': 0.12, 'user_pref': 0.14, 'significant': 0.05, 'engagement': 0.10,
    },
    'archetype_heavy': {
        'archetype_sim': 0.35, 'semantic_sim': 0.14, 'route_sim': 0.10, 'flow_prob': 0.08,
        'dominant_ctx': 0.10, 'user_pref': 0.08, 'significant': 0.05, 'engagement': 0.04,
        'rule_coverage': 0.06,
    },
    'semantic_heavy': {
        'archetype_sim': 0.10, 'semantic_sim': 0.35, 'route_sim': 0.10, 'flow_prob': 0.08,
        'dominant_ctx': 0.10, 'user_pref': 0.10, 'significant': 0.04, 'engagement': 0.04,
        'rule_coverage': 0.09,
    },
    'route_heavy': {
        'archetype_sim': 0.10, 'semantic_sim': 0.10, 'route_sim': 0.30, 'flow_prob': 0.20,
        'dominant_ctx': 0.08, 'user_pref': 0.08, 'significant': 0.05, 'engagement': 0.04,
        'rule_coverage': 0.05,
    },
    'rules_heavy': {
        'archetype_sim': 0.10, 'semantic_sim': 0.10, 'route_sim': 0.08, 'flow_prob': 0.06,
        'dominant_ctx': 0.10, 'user_pref': 0.08, 'significant': 0.05, 'engagement': 0.04,
        'rule_coverage': 0.39,
    },
    'context_heavy': {
        'archetype_sim': 0.08, 'semantic_sim': 0.08, 'route_sim': 0.06, 'flow_prob': 0.06,
        'dominant_ctx': 0.28, 'user_pref': 0.20, 'significant': 0.08, 'engagement': 0.10,
        'rule_coverage': 0.06,
    },
    'equal': {
        'archetype_sim': 1/9, 'semantic_sim': 1/9, 'route_sim': 1/9, 'flow_prob': 1/9,
        'dominant_ctx': 1/9, 'user_pref': 1/9, 'significant': 1/9, 'engagement': 1/9,
        'rule_coverage': 1/9,
    },
    'no_archetype': {
        'archetype_sim': 0.00, 'semantic_sim': 0.26, 'route_sim': 0.18, 'flow_prob': 0.12,
        'dominant_ctx': 0.14, 'user_pref': 0.12, 'significant': 0.06, 'engagement': 0.05,
        'rule_coverage': 0.07,
    },
    'no_route': {
        'archetype_sim': 0.26, 'semantic_sim': 0.26, 'route_sim': 0.00, 'flow_prob': 0.00,
        'dominant_ctx': 0.16, 'user_pref': 0.14, 'significant': 0.08, 'engagement': 0.05,
        'rule_coverage': 0.05,
    },
    'no_rules': {
        'archetype_sim': 0.24, 'semantic_sim': 0.24, 'route_sim': 0.16, 'flow_prob': 0.10,
        'dominant_ctx': 0.10, 'user_pref': 0.08, 'significant': 0.05, 'engagement': 0.03,
    },
    'similarity_only': {
        'archetype_sim': 0.28, 'semantic_sim': 0.28, 'route_sim': 0.20, 'flow_prob': 0.00,
        'dominant_ctx': 0.00, 'user_pref': 0.12, 'significant': 0.06, 'engagement': 0.06,
    },
}

k_values = [5, 10, 20]
protocols = ['trajectory_split', 'leave_k_out']

weight_results = {}
t_total = time.time()

for config_name, weights in weight_configs.items():
    for protocol in protocols:
        t0 = time.time()
        leave_k = 1 if protocol == 'leave_k_out' else 5
        result = urec.evaluate_recommendations(
            method='unified', k_values=k_values,
            protocol=protocol, split_ratio=0.7, leave_k=leave_k,
            signal_weights=weights, min_seq_length=3
        )
        weight_results[(protocol, config_name)] = result
        print(f'  {config_name:20s} {protocol:18s}  '
              f'R@5={result["recall@5"]:.4f}  R@10={result["recall@10"]:.4f}  '
              f'R@20={result["recall@20"]:.4f}  MRR={result["mrr"]:.4f}  '
              f'N={result["n_evaluated"]}  [{time.time()-t0:.1f}s]')

print(f'\nTotal: {time.time()-t_total:.1f}s')

In [ ]:
# Results table
rows = []
for (protocol, config), result in sorted(weight_results.items()):
    rows.append({
        'Protocol': protocol,
        'Config': config,
        'N': result['n_evaluated'],
        'Recall@5': result.get('recall@5', 0),
        'Recall@10': result.get('recall@10', 0),
        'Recall@20': result.get('recall@20', 0),
        'MRR': result.get('mrr', 0),
        'MAP': result.get('map', 0),
    })

df_weights = pd.DataFrame(rows)
print('Signal Weight Sensitivity (min_seq_length=3):')
display(df_weights.style.format({
    'Recall@5': '{:.4f}', 'Recall@10': '{:.4f}', 'Recall@20': '{:.4f}',
    'MRR': '{:.4f}', 'MAP': '{:.4f}',
}).highlight_max(subset=['Recall@5', 'Recall@10', 'Recall@20', 'MRR', 'MAP'],
                 color='#d4edda', axis=0))

In [ ]:
# Heatmap: config x metric, per protocol
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, protocol in zip(axes, protocols):
    proto_rows = df_weights[df_weights['Protocol'] == protocol].copy()
    proto_rows = proto_rows.set_index('Config')[['Recall@5', 'Recall@10', 'Recall@20', 'MRR', 'MAP']]

    sns.heatmap(proto_rows, annot=True, fmt='.4f', cmap='YlGn',
                ax=ax, linewidths=0.5, vmin=0)
    ax.set_title(protocol.replace('_', ' ').title())
    ax.set_ylabel('')

plt.suptitle('Signal Weight Sensitivity', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'signal_weight_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart: Recall@10 across configs, grouped by protocol
fig, ax = plt.subplots(figsize=(16, 5))

config_order = ['default', 'archetype_heavy', 'semantic_heavy', 'route_heavy', 'rules_heavy',
                'context_heavy', 'equal', 'no_archetype', 'no_route', 'no_rules', 'similarity_only']
x = np.arange(len(config_order))
width = 0.35

for i, protocol in enumerate(protocols):
    vals = [weight_results.get((protocol, c), {}).get('recall@10', 0) for c in config_order]
    offset = -width/2 + i * width
    bars = ax.bar(x + offset, vals, width, label=protocol.replace('_', ' ').title(),
                  alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in config_order], fontsize=8)
ax.set_ylabel('Recall@10')
ax.set_title('Recall@10 by Signal Weight Configuration')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'signal_weight_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Split Ratio Sensitivity

How does the train/test split ratio affect unified recommender performance? Tested with two eligibility thresholds (>=3 and >=5 POIs).

In [ ]:
split_ratios = [0.4, 0.5, 0.6, 0.7, 0.8]
min_lengths = [3, 5]

split_results = {}
t_total = time.time()

for min_len in min_lengths:
    eligible_n = sum(1 for s in urec.sequences.values() if len(s) >= min_len)
    print(f'\n=== min_seq_length={min_len} ({eligible_n} trajectories) ===')
    for ratio in split_ratios:
        t0 = time.time()
        result = urec.evaluate_recommendations(
            method='unified', k_values=k_values,
            protocol='trajectory_split', split_ratio=ratio,
            min_seq_length=min_len
        )
        split_results[(min_len, ratio)] = result
        ci10 = result.get('recall@10_ci', (0,0))
        print(f'  ratio={ratio:.1f}  N={result["n_evaluated"]:3d}  '
              f'R@5={result["recall@5"]:.4f}  '
              f'R@10={result["recall@10"]:.4f} ({ci10[0]:.4f}-{ci10[1]:.4f})  '
              f'MRR={result["mrr"]:.4f}  [{time.time()-t0:.1f}s]')

    # Also test leave-1-out
    t0 = time.time()
    result = urec.evaluate_recommendations(
        method='unified', k_values=k_values,
        protocol='leave_k_out', leave_k=1,
        min_seq_length=min_len
    )
    split_results[(min_len, 'leave_1')] = result
    print(f'  leave-1   N={result["n_evaluated"]:3d}  '
          f'R@5={result["recall@5"]:.4f}  '
          f'R@10={result["recall@10"]:.4f}  '
          f'MRR={result["mrr"]:.4f}  [{time.time()-t0:.1f}s]')

print(f'\nTotal: {time.time()-t_total:.1f}s')

In [ ]:
# Split Ratio Sensitivity — results table (one per min_seq_length group)
for min_len in min_lengths:
    rows = []
    best_r10_ratio, best_mrr_ratio = None, None
    best_r10, best_mrr = -1, -1

    for ratio in split_ratios:
        r = split_results.get((min_len, ratio))
        if not r:
            continue
        ci_r10 = r.get('recall@10_ci', (0, 0))
        ci_mrr = r.get('mrr_ci', (0, 0))
        rows.append({
            'Split Ratio': ratio,
            'N': r['n_evaluated'],
            'R@5': r.get('recall@5', 0),
            'R@10': r.get('recall@10', 0),
            'R@10 CI': f"[{ci_r10[0]:.3f}, {ci_r10[1]:.3f}]",
            'R@20': r.get('recall@20', 0),
            'MRR': r.get('mrr', 0),
            'MRR CI': f"[{ci_mrr[0]:.3f}, {ci_mrr[1]:.3f}]" if ci_mrr != (0, 0) else '\u2014',
        })
        if r.get('recall@10', 0) > best_r10:
            best_r10 = r['recall@10']
            best_r10_ratio = ratio
        if r.get('mrr', 0) > best_mrr:
            best_mrr = r['mrr']
            best_mrr_ratio = ratio

    # Add leave-1-out as reference row
    r_l1 = split_results.get((min_len, 'leave_1'))
    if r_l1:
        rows.append({
            'Split Ratio': 'leave-1-out',
            'N': r_l1['n_evaluated'],
            'R@5': r_l1.get('recall@5', 0),
            'R@10': r_l1.get('recall@10', 0),
            'R@10 CI': '\u2014',
            'R@20': r_l1.get('recall@20', 0),
            'MRR': r_l1.get('mrr', 0),
            'MRR CI': '\u2014',
        })

    n_eligible = sum(1 for s in urec.sequences.values() if len(s) >= min_len)
    print(f"\n{'='*70}")
    print(f"  Sequences with >= {min_len} POIs  (N = {n_eligible})")
    print(f"  Best R@10: ratio={best_r10_ratio} ({best_r10:.4f})")
    print(f"  Best MRR:  ratio={best_mrr_ratio} ({best_mrr:.4f})")
    print(f"{'='*70}")

    df = pd.DataFrame(rows)
    display(df.style
        .format({
            'R@5': '{:.4f}', 'R@10': '{:.4f}', 'R@20': '{:.4f}',
            'MRR': '{:.4f}',
        })
        .highlight_max(
            subset=['R@5', 'R@10', 'R@20', 'MRR'], color='#d4edda', axis=0)
        .highlight_min(
            subset=['R@5', 'R@10', 'R@20', 'MRR'], color='#f8d7da', axis=0)
        .set_caption(f'Split Ratio Sensitivity (min_seq_length={min_len})')
    )


In [ ]:
# Line plots with confidence intervals
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {3: '#e74c3c', 5: '#3498db'}
markers = {3: 'o', 5: 's'}

for min_len in min_lengths:
    ratios_plot = []
    recall_vals = []
    mrr_vals = []
    recall_lo, recall_hi = [], []
    mrr_lo, mrr_hi = [], []

    for ratio in split_ratios:
        r = split_results.get((min_len, ratio))
        if r:
            ratios_plot.append(ratio)
            recall_vals.append(r['recall@10'])
            mrr_vals.append(r['mrr'])
            ci_r = r.get('recall@10_ci', (0, 0))
            ci_m = r.get('mrr_ci', (0, 0))
            recall_lo.append(r['recall@10'] - ci_r[0])
            recall_hi.append(ci_r[1] - r['recall@10'])
            mrr_lo.append(r['mrr'] - ci_m[0])
            mrr_hi.append(ci_m[1] - r['mrr'])

    c = colors[min_len]
    m = markers[min_len]
    n = sum(1 for s in urec.sequences.values() if len(s) >= min_len)
    label = f'>={min_len} POIs (N={n})'

    ax1.errorbar(ratios_plot, recall_vals, yerr=[recall_lo, recall_hi],
                 fmt=f'{m}-', color=c, label=label, linewidth=2,
                 markersize=7, capsize=4)
    ax2.errorbar(ratios_plot, mrr_vals, yerr=[mrr_lo, mrr_hi],
                 fmt=f'{m}-', color=c, label=label, linewidth=2,
                 markersize=7, capsize=4)

ax1.set_xlabel('Split Ratio (train fraction)')
ax1.set_ylabel('Recall@10')
ax1.set_title('Recall@10 vs Split Ratio')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Split Ratio (train fraction)')
ax2.set_ylabel('MRR')
ax2.set_title('MRR vs Split Ratio')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Unified Recommender: Split Ratio Sensitivity', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'unified_split_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Signal Contribution Profiling

For each signal, measure how many recommendations it contributes to and its average normalized score across all test trajectories.

In [ ]:
# Profile signal activity across recommendations
eligible = [t for t in urec.sequences if len(urec.sequences[t]) >= 3]
signal_names = list(UNIFIED_WEIGHTS.keys())

signal_stats = {s: {'active': 0, 'total_score': 0.0, 'top1_count': 0} for s in signal_names}
total_recs = 0

for traj_uri in eligible:
    recs = urec.recommend(traj_uri, top_k=10)
    for rec in recs:
        total_recs += 1
        signals = rec.get('signals', {})
        for s in signal_names:
            val = signals.get(s, 0)
            if val > 0:
                signal_stats[s]['active'] += 1
                signal_stats[s]['total_score'] += val
        # Which signal has highest weighted contribution?
        if signals:
            weighted = {s: UNIFIED_WEIGHTS.get(s, 0) * signals.get(s, 0) for s in signal_names}
            top_signal = max(weighted, key=weighted.get)
            signal_stats[top_signal]['top1_count'] += 1

print(f'Analyzed {total_recs} recommendations from {len(eligible)} trajectories\n')
print(f'{"Signal":20s} {"Active%":>8s} {"Avg Score":>10s} {"Top-1%":>8s} {"Weight":>8s}')
print('-' * 58)
for s in signal_names:
    st = signal_stats[s]
    active_pct = st['active'] / total_recs * 100 if total_recs else 0
    avg_score = st['total_score'] / st['active'] if st['active'] else 0
    top1_pct = st['top1_count'] / total_recs * 100 if total_recs else 0
    print(f'{s:20s} {active_pct:7.1f}% {avg_score:9.4f} {top1_pct:7.1f}% {UNIFIED_WEIGHTS[s]:7.2f}')

In [ ]:
# Visualization: signal activity and dominance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

active_pcts = [signal_stats[s]['active'] / total_recs * 100 for s in signal_names]
top1_pcts = [signal_stats[s]['top1_count'] / total_recs * 100 for s in signal_names]
weights_pcts = [UNIFIED_WEIGHTS[s] * 100 for s in signal_names]

x = np.arange(len(signal_names))
width = 0.35

ax1.barh(x, active_pcts, height=0.6, color='#3498db', alpha=0.8)
ax1.set_yticks(x)
ax1.set_yticklabels([s.replace('_', ' ') for s in signal_names], fontsize=9)
ax1.set_xlabel('% of recommendations where signal > 0')
ax1.set_title('Signal Activity Rate')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(active_pcts):
    ax1.text(v + 0.5, i, f'{v:.0f}%', va='center', fontsize=8)

ax2.barh(x - width/2, top1_pcts, height=0.35, color='#e74c3c', alpha=0.8, label='Top-1 dominant')
ax2.barh(x + width/2, weights_pcts, height=0.35, color='#95a5a6', alpha=0.8, label='Assigned weight')
ax2.set_yticks(x)
ax2.set_yticklabels([s.replace('_', ' ') for s in signal_names], fontsize=9)
ax2.set_xlabel('%')
ax2.set_title('Signal Dominance vs Assigned Weight')
ax2.legend(fontsize=9)
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Signal Contribution Profile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'signal_contribution_profile.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Drop-One Ablation

For each signal, set its weight to 0 and redistribute proportionally. Measures the marginal contribution of each signal.

In [ ]:
# Drop-one ablation: zero each signal, redistribute remaining
ablation_results = {}
t_total = time.time()

# Baseline (all signals)
baseline = urec.evaluate_recommendations(
    method='unified', k_values=k_values,
    protocol='trajectory_split', split_ratio=0.7,
    min_seq_length=3
)
ablation_results['all_signals'] = baseline
print(f'  {"all_signals":20s}  R@10={baseline["recall@10"]:.4f}  MRR={baseline["mrr"]:.4f}')

for drop_signal in signal_names:
    # Zero the dropped signal and redistribute
    remaining = {s: w for s, w in UNIFIED_WEIGHTS.items() if s != drop_signal}
    total_remaining = sum(remaining.values())
    ablated_weights = {s: w / total_remaining for s, w in remaining.items()}
    ablated_weights[drop_signal] = 0.0

    result = urec.evaluate_recommendations(
        method='unified', k_values=k_values,
        protocol='trajectory_split', split_ratio=0.7,
        signal_weights=ablated_weights, min_seq_length=3
    )
    ablation_results[f'drop_{drop_signal}'] = result

    delta_r = result['recall@10'] - baseline['recall@10']
    delta_m = result['mrr'] - baseline['mrr']
    sign_r = '+' if delta_r >= 0 else ''
    sign_m = '+' if delta_m >= 0 else ''
    print(f'  drop {drop_signal:20s}  R@10={result["recall@10"]:.4f} ({sign_r}{delta_r:.4f})  '
          f'MRR={result["mrr"]:.4f} ({sign_m}{delta_m:.4f})')

print(f'\nTotal: {time.time()-t_total:.1f}s')

In [ ]:
# Ablation impact chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

base_r10 = baseline['recall@10']
base_mrr = baseline['mrr']

deltas_r = []
deltas_m = []
for s in signal_names:
    r = ablation_results.get(f'drop_{s}', {})
    deltas_r.append(r.get('recall@10', 0) - base_r10)
    deltas_m.append(r.get('mrr', 0) - base_mrr)

colors_r = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas_r]
colors_m = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas_m]

y = np.arange(len(signal_names))

ax1.barh(y, deltas_r, color=colors_r, alpha=0.8)
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.set_yticks(y)
ax1.set_yticklabels([s.replace('_', ' ') for s in signal_names], fontsize=9)
ax1.set_xlabel('Delta Recall@10 when dropped')
ax1.set_title(f'Recall@10 Impact (baseline={base_r10:.4f})')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(deltas_r):
    ax1.text(v + 0.001 if v >= 0 else v - 0.001, i,
             f'{v:+.4f}', va='center', ha='left' if v >= 0 else 'right', fontsize=8)

ax2.barh(y, deltas_m, color=colors_m, alpha=0.8)
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_yticks(y)
ax2.set_yticklabels([s.replace('_', ' ') for s in signal_names], fontsize=9)
ax2.set_xlabel('Delta MRR when dropped')
ax2.set_title(f'MRR Impact (baseline={base_mrr:.4f})')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(deltas_m):
    ax2.text(v + 0.001 if v >= 0 else v - 0.001, i,
             f'{v:+.4f}', va='center', ha='left' if v >= 0 else 'right', fontsize=8)

plt.suptitle('Drop-One Ablation: Signal Marginal Value', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'signal_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()